In [ ]:
!pip install torch scikit-learn

In [ ]:
# --- Tools we need for our exercise---
import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1) LOAD THE DATA
data     = fetch_california_housing()
features = data.data  # 8 input numbers for each house
prices   = data.target.reshape(-1, 1)  # Reshape per lo scaler

print(f"Number of samples: {len(prices)}")
print(f"Number of features: {features.shape[1]}")
print(f"Feature names: {data.feature_names}")

In [ ]:
# 2) SPLIT THE DATA (aggiungiamo anche validation set)

# We divide the dataset into 3 parts:
# training (learn), validation (check progress), test (final evaluation)

train_features, temp_features, train_prices, temp_prices = train_test_split(
    features, prices, test_size=0.3, random_state=42
)

val_features, test_features, val_prices, test_prices = train_test_split(
    temp_features, temp_prices, test_size=0.5, random_state=42
)

# Show how many samples are in each group
print("Training samples:", len(train_prices))
print("Validation samples:", len(val_prices))
print("Test samples:", len(test_prices))


In [ ]:
# 3) SCALE THE DATA

# We rescale the data so all numbers are comparable
# (mean = 0, standard deviation = 1)

feature_scaler = StandardScaler()
price_scaler = StandardScaler()

# Scale input data (house features)
train_features = feature_scaler.fit_transform(train_features)
val_features   = feature_scaler.transform(val_features)
test_features  = feature_scaler.transform(test_features)

# Scale output data (house prices)
train_prices = price_scaler.fit_transform(train_prices)
val_prices   = price_scaler.transform(val_prices)
test_prices  = price_scaler.transform(test_prices)

print("Data scaled: all values now have similar size")


In [ ]:
# 4) CONVERT TO PYTORCH TENSORS
train_features = torch.tensor(train_features, dtype=torch.float32)
train_prices   = torch.tensor(train_prices, dtype=torch.float32)

val_features = torch.tensor(val_features, dtype=torch.float32)
val_prices = torch.tensor(val_prices, dtype=torch.float32)

test_features  = torch.tensor(test_features, dtype=torch.float32)
test_prices = torch.tensor(test_prices, dtype=torch.float32)


In [ ]:
# 5) MODELLO CON DROPOUT PER REGOLARIZZAZIONE
class HousePriceModel(nn.Module):
    def __init__(self, n_features, dropout_rate=0.1):
        super().__init__()
        
        # Modello con dropout per regolarizzazione
        self.network = nn.Sequential(
            nn.Linear(n_features, 256),    # First layer from input features to 256 neurons
            nn.ReLU(),                     # Non-linear activation
            nn.Dropout(dropout_rate),      # Dropout after first hidden layer

            nn.Linear(256, 128),           # Second  layer
            nn.ReLU(),                     # Non-linear activation
            nn.Dropout(dropout_rate),      # Dropout after second  layer

            nn.Linear(128, 64),            # Third  layer
            nn.ReLU(),
            nn.Dropout(dropout_rate),      # Dropout after third layer

            nn.Linear(64, 32),             # Fourth layer
            nn.ReLU(),                     # Non-linear activation

            nn.Linear(32, 16),             # Fifth layer
            nn.ReLU(),                     # Non-linear activation

            nn.Linear(16, 1)               # Output layer (no dropout here)
        )

    def forward(self, x):
        return self.network(x)


model = HousePriceModel(train_features.shape[1], dropout_rate=0.1)
print(model)


In [ ]:
# We measure how wrong the predictions are
loss_function = nn.MSELoss()

# How big each learning step is
learning_rate = 0.005

# Weight decay (L2 regularization) - penalizes large weights
weight_decay = 1e-4

# Tool that updates the model to make predictions better
# Added weight_decay for L2 regularization
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)



# How many times we repeat the learning process
number_of_epochs = 2000

for epoch in range(number_of_epochs):

    # --- TRAINING ---
    model.train()                         # learning mode (dropout active)
    predicted_prices = model(train_features)
    train_loss = loss_function(predicted_prices, train_prices)

    optimizer.zero_grad()                 # reset old gradients
    train_loss.backward()                 # understand mistakes
    optimizer.step()                      # improve the model

    # --- VALIDATION ---
    model.eval()                          # evaluation mode (dropout disabled)
    with torch.no_grad():
        val_predictions = model(val_features)
        val_loss = loss_function(val_predictions, val_prices)

    # Print progress sometimes
    if epoch % 50 == 0:
        print(
            f"Epoch {epoch} | "
            f"Train error: {train_loss.item():.4f} | "
            f"Validation error: {val_loss.item():.4f}"
        )

In [ ]:
# 7) EVALUATE ON TEST SET

# Switch the model to evaluation mode
model.eval()
with torch.no_grad():
    test_predictions_scaled = model(test_features).cpu().numpy()

# De-scale predictions and targets to original units (prices in $100k)
test_predictions = price_scaler.inverse_transform(test_predictions_scaled)
test_prices_original = price_scaler.inverse_transform(test_prices.cpu().numpy())

# Calculate metrics on original values
mae = np.mean(np.abs(test_predictions - test_prices_original))
rmse = np.sqrt(np.mean((test_predictions - test_prices_original) ** 2))
mape = np.mean(np.abs((test_predictions - test_prices_original) / test_prices_original)) * 100

# Calculate R² score
ss_res = np.sum((test_prices_original - test_predictions) ** 2)
ss_tot = np.sum((test_prices_original - np.mean(test_prices_original)) ** 2)
r2_score = 1 - (ss_res / ss_tot)

# Display results in a clear format
print("=" * 60)
print("                    FINAL RESULTS")
print("=" * 60)
print()
print("Model Performance Metrics:")
print("-" * 40)
print(f"   Mean Absolute Error:     ${mae * 100000:>12,.0f}")
print(f"   Mean Absolute % Error:        {mape:>12.2f}%")